# Module 13 Lab - Building ML Pipelines

**Objective:** To understand the importance of `scikit-learn` **Pipelines** for creating robust, reproducible, and professional machine learning workflows.**In this lab, you will refactor code from a previous lab into a clean, professional `Pipeline` object.**

## Part 1: Why Use Pipelines?

**Concept:** As you've seen, a typical ML workflow involves multiple steps: loading data, cleaning it, splitting it, preprocessing features (scaling, encoding), and finally, training a model. Managing all these steps separately can be messy and error-prone.

**Data Leakage:** A major risk of manual preprocessing is **data leakage**. This happens when information from the test set accidentally "leaks" into the training process. For example, if you calculate the mean for scaling using the *entire* dataset before splitting, the model has already "seen" the test data, leading to overly optimistic performance estimates.**A `scikit-learn` Pipeline solves these problems by:

1.  **Encapsulating** all workflow steps into a single object.

2.  **Preventing Data Leakage:** It ensures that preprocessing steps are fitted *only* on the training data during cross-validation or when calling `.fit()`.
3.  **Improving Reproducibility:** The entire workflow is saved as one object, making it easy to reuse and deploy.

## Part 2: The "Manual" Way (What We Did Before)

Let's revisit the Titanic dataset and the steps we took to prepare the data and train a model. This code should look familiar. Notice how many separate objects and steps there are.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


# Load the data
df = pd.read_csv(
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
)


# Basic feature engineering and cleaning
df["Age"].fillna(df["Age"].median(), inplace=True)
df["Embarked"].fillna(df["Embarked"].mode()[0], inplace=True)

df.drop("Cabin", axis=1, inplace=True)


# Separate features and target
X = df.drop(
    ["Survived", "Name", "Ticket", "PassengerId"],
    axis=1
)

y = df["Survived"]


# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# Identify feature types
numeric_features = ["Age", "Fare", "SibSp", "Parch"]
categorical_features = ["Pclass", "Sex", "Embarked"]


# Scale the numerical features
scaler = StandardScaler()

X_train_scaled_num = scaler.fit_transform(
    X_train[numeric_features]
)

X_test_scaled_num = scaler.transform(
    X_test[numeric_features]
)


# Encode the categorical features
encoder = OneHotEncoder(handle_unknown="ignore")

X_train_encoded_cat = encoder.fit_transform(
    X_train[categorical_features]
)

X_test_encoded_cat = encoder.transform(
    X_test[categorical_features]
)


# Combine numerical and categorical features
X_train_processed = np.hstack(
    (
        X_train_scaled_num,
        X_train_encoded_cat.toarray()
    )
)

X_test_processed = np.hstack(
    (
        X_test_scaled_num,
        X_test_encoded_cat.toarray()
    )
)


# Train the model
model = RandomForestClassifier(random_state=42)

model.fit(
    X_train_processed,
    y_train
)


# Make predictions
y_pred = model.predict(X_test_processed)


# Evaluate the model
print(
    f"Accuracy (Manual Method): "
    f"{accuracy_score(y_test, y_pred):.2%}"
)

/tmp/ipykernel_3397/1790633052.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Age"].fillna(df["Age"].median(), inplace=True)
/tmp/ipykernel_3397/1790633052.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', 

Accuracy (Manual Method): 82.68%


## Part 3: The "Pipeline" Way

Now, let's do the exact same thing but encapsulate all the preprocessing steps into a single `Pipeline`.

**Your Task:** Use `make_pipeline` and `make_column_transformer` to build a complete workflow. This is the modern, professional way to build models in `scikit-learn`.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer

# Reload the data to start fresh
df = pd.read_csv(
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
)

df.drop(
    ["Cabin", "Name", "Ticket", "PassengerId"],
    axis=1,
    inplace=True
)

X = df.drop("Survived", axis=1)
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# 1. Create a pipeline for numeric features
numeric_transformer = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)


# 2. Create a pipeline for categorical features
categorical_transformer = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)


# 3. Apply each pipeline to the correct columns
preprocessor = make_column_transformer(
    (
        numeric_transformer,
        ["Age", "Fare", "SibSp", "Parch"]
    ),
    (
        categorical_transformer,
        ["Pclass", "Sex", "Embarked"]
    )
)


# 4. Build the complete machine-learning pipeline
pipeline_model = make_pipeline(
    preprocessor,
    RandomForestClassifier(random_state=42)
)


# 5. Train the complete pipeline
pipeline_model.fit(
    X_train,
    y_train
)


# 6. Make predictions
y_pred_pipeline = pipeline_model.predict(X_test)


# 7. Evaluate the model
print(
    f"Accuracy (Pipeline Method): "
    f"{accuracy_score(y_test, y_pred_pipeline):.2%}"
)

Accuracy (Pipeline Method): 82.68%


## 📝 Reflective Knowledge Check

**Instructions:** Answer the following questions in this markdown cell.

1.  **Code Comparison:** Look at the "Manual Way" versus the "Pipeline Way". What are the three biggest advantages you see in using the Pipeline approach?

2.  **Data Leakage Explained:** In the manual code, we used `scaler.fit_transform()` on the training data but only `scaler.transform()` on the test data. Why was this distinction crucial? How does the Pipeline automatically handle this for you?

3.  **Extending the Pipeline:** Imagine you wanted to add a PCA step to reduce dimensionality *after* scaling and encoding but *before* the RandomForestClassifier. How would you modify your `final_pipeline` object to include this step? (You don't need to write the full code, just describe where you would add `PCA()`.)

4.  **Real-World Value:** You are handing your model over to another team to deploy into a web application. Why is giving them the single `final_pipeline` object much safer and more reliable than giving them the 5 separate objects (`scaler`, `encoder`, `model`, etc.) from the manual approach?

**[ENTER YOUR ANSWERS HERE]**

1.The three biggest advantages I see in using the Pipeline approach are cleaner code, reduced risk of mistakes, and easier reuse. In the manual method, scaling, encoding, combining the processed features, and training the model all had to be handled separately. The Pipeline method places those steps into one organized workflow.

It also makes it harder to accidentally apply different preprocessing to the training and testing data. Finally, the entire process can be saved, reused, and deployed as one object. In my results, both approaches produced the same accuracy of 82.68%, showing that the Pipeline completed the same work with a cleaner and more reliable structure.

2.Using scaler.fit_transform() on the training data was important because fit_transform() learns information from the data, such as the mean and standard deviation, and then applies the scaling. The test data must not influence those learned values because it is supposed to represent unseen data.

That is why the test data only used: scaler.transform()

This applied the values already learned from the training data without recalculating them. If the scaler had been fitted using the test data, information from the test set would leak into the training process and could make the model’s results appear better than they really are.

The Pipeline handles this automatically. When final_pipeline.fit(X_train, y_train) is called, the preprocessing steps are fitted only on the training data. When final_pipeline.predict(X_test) is called, the Pipeline uses the already fitted preprocessing steps to transform the test data.

3.I would add PCA() after the preprocessor and before the RandomForestClassifier. The order would be:

preprocessor → PCA → RandomForestClassifier

For example, the final Pipeline would conceptually contain:

final_pipeline = make_pipeline(
    
    preprocessor,
    PCA(),
    RandomForestClassifier(random_state=42)
)

This allows the data to be cleaned, scaled, and encoded first. PCA would then reduce the dimensions of the processed data before it is passed to the Random Forest model.


4. Giving another team the single final_pipeline object is safer because it contains the exact preprocessing and model steps used during training. The deployment team can provide raw input data directly to the Pipeline, and it will automatically impute missing values, scale numerical features, encode categorical features, and make a prediction in the correct order.

With five separate objects, the team could accidentally skip a step, apply the steps in the wrong order, use the wrong columns, or refit a scaler or encoder on new data. These mistakes could cause incorrect predictions or errors in the application. A single Pipeline reduces those risks and makes the model easier to save, load, test, and deploy consistently.